# HAM10000 — Retrain Top-3 Models PER FILTER (Kaggle Notebook, GPU)



In [ ]:
# ── Installs (usually already present in the Kaggle python image) ────────
try:
    import xgboost, cv2  # noqa
except ImportError:
    import sys
    !{sys.executable} -m pip install -q xgboost opencv-python-headless


In [ ]:
# ── Imports & global config ──────────────────────────────────────────────
import os, glob, copy, time, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import cv2
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.models import ResNet101_Weights, DenseNet121_Weights

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import xgboost as xgb

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE, "-", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

IMG_SIZE     = 224
BATCH_SIZE   = 64
NUM_WORKERS  = min(os.cpu_count() or 2, 4)
MAX_EPOCHS   = 15
PATIENCE     = 6
LR           = 5e-5
WEIGHT_DECAY = 1e-5

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

WORK_DIR = Path("/kaggle/working")
WORK_DIR.mkdir(exist_ok=True)


Device: cuda - Tesla T4


In [ ]:
import kagglehub
path = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")
print("Path:", path)

# Find CSV
csv_path = None
for root, dirs, files in os.walk(path):
    for f in files:
        if 'metadata' in f.lower() and f.endswith('.csv'):
            csv_path = os.path.join(root, f); break
    if csv_path: break
print("CSV:", csv_path)

# Find all images
img_map = {}
for root, dirs, files in os.walk(path):
    for f in files:
        if f.lower().endswith('.jpg'):
            img_map[os.path.splitext(f)[0]] = os.path.join(root, f)
print(f"Images found: {len(img_map)}")

# Load + clean
df          = pd.read_csv(csv_path)
df          = df.drop_duplicates(subset='lesion_id', keep='first')
df          = df[df['image_id'].isin(img_map)].reset_index(drop=True)
df['path']  = df['image_id'].map(img_map)
CLASSES     = sorted(df['dx'].unique())
NUM_CLASSES = len(CLASSES)
cls2idx     = {c:i for i,c in enumerate(CLASSES)}
df['label'] = df['dx'].map(cls2idx)

print(f"Samples:{len(df)}  Classes({NUM_CLASSES}): {CLASSES}")
print(df['dx'].value_counts())

# 80/10/10 split
train_df, test_df = train_test_split(df, test_size=0.20,
                    stratify=df['label'], random_state=42)
train_df, val_df  = train_test_split(train_df, test_size=0.125,
                    stratify=train_df['label'], random_state=42)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)
print(f"Train:{len(train_df)}  Val:{len(val_df)}  Test:{len(test_df)}")

Using Colab cache for faster access to the 'skin-cancer-mnist-ham10000' dataset.
Path: /kaggle/input/skin-cancer-mnist-ham10000
CSV: /kaggle/input/skin-cancer-mnist-ham10000/HAM10000_metadata.csv
Images found: 10015
Samples:7470  Classes(7): ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
dx
nv       5403
bkl       727
mel       614
bcc       327
akiec     228
vasc       98
df         73
Name: count, dtype: int64
Train:5229  Val:747  Test:1494


In [ ]:
# ── Load metadata + map images (same logic as the original notebook) ──────
df = pd.read_csv(csv_path)

label_names = {
    "akiec": "Actinic keratoses", "bcc": "Basal cell carcinoma",
    "bkl": "Benign keratosis-like lesions", "df": "Dermatofibroma",
    "mel": "Melanoma", "nv": "Melanocytic nevi", "vasc": "Vascular lesions",
}
df["label_name"] = df["dx"].map(label_names)

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
image_map = {}
for root, _, files in os.walk("/kaggle/input"):
    for f in files:
        if Path(f).suffix.lower() in image_extensions:
            image_map[Path(f).stem] = str(Path(root) / f)

df["image_path"] = df["image_id"].map(image_map)
df_valid = df[df["image_path"].notna()].copy()
print("Valid images:", len(df_valid))

CLASSES = sorted(df_valid["dx"].unique().tolist())
NUM_CLASSES = len(CLASSES)
label_to_idx = {c: i for i, c in enumerate(CLASSES)}
print("Classes:", CLASSES)

Valid images: 10015
Classes: ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']


In [ ]:
# ── Lesion-level leakage-free split (identical to the original notebook: SEED=42) ──
lesion_df = df_valid[["lesion_id", "dx"]].drop_duplicates(subset=["lesion_id"]).reset_index(drop=True)

train_lesions, temp_lesions = train_test_split(
    lesion_df, test_size=0.20, stratify=lesion_df["dx"], random_state=SEED
)
val_lesions, test_lesions = train_test_split(
    temp_lesions, test_size=0.50, stratify=temp_lesions["dx"], random_state=SEED
)

train_df = df_valid[df_valid["lesion_id"].isin(train_lesions["lesion_id"])].copy()
val_df   = df_valid[df_valid["lesion_id"].isin(val_lesions["lesion_id"])].copy()
test_df  = df_valid[df_valid["lesion_id"].isin(test_lesions["lesion_id"])].copy()

assert set(train_df.lesion_id).isdisjoint(val_df.lesion_id)
assert set(train_df.lesion_id).isdisjoint(test_df.lesion_id)
assert set(val_df.lesion_id).isdisjoint(test_df.lesion_id)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

# Class weights (imbalance) — depend only on labels, so computed once, shared by all filters
train_labels_all = train_df["dx"].map(label_to_idx).values
class_weights = compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=train_labels_all)
class_weights_t = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)
print(dict(zip(CLASSES, class_weights.round(3))))


Train: 8020 | Val: 1005 | Test: 990
{'akiec': np.float64(4.323), 'bcc': np.float64(2.767), 'bkl': np.float64(1.3), 'df': np.float64(13.169), 'mel': np.float64(1.28), 'nv': np.float64(0.213), 'vasc': np.float64(10.416)}


In [ ]:
# ── Filters (OpenCV, applied to the resized uint8 RGB array) ─────────────
def f_none(img):       return img
def f_average(img):    return cv2.blur(img, (5, 5))
def f_gaussian(img):   return cv2.GaussianBlur(img, (5, 5), 0)
def f_median(img):     return cv2.medianBlur(img, 5)

def f_sharpen(img):
    kernel = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], dtype=np.float32)
    out = cv2.filter2D(img, -1, kernel)
    return np.clip(out, 0, 255).astype(np.uint8)

def f_sobel(img):
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    gx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    mag = np.sqrt(gx ** 2 + gy ** 2)
    mag = (mag / (mag.max() + 1e-8) * 255).astype(np.uint8)
    return np.stack([mag, mag, mag], axis=-1)

FILTERS = {
    "No Filter": f_none,
    "Average":   f_average,
    "Gaussian":  f_gaussian,
    "Median":    f_median,
    "Sharpening": f_sharpen,
    "Sobel":     f_sobel,
}


In [ ]:
# ── Pre-materialize filtered images once per filter (avoids re-filtering every epoch) ──
def build_array_split(dataframe, filter_fn):
    n = len(dataframe)
    imgs = np.zeros((n, IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
    labels = np.zeros(n, dtype=np.int64)
    for i, row in enumerate(dataframe.itertuples(index=False)):
        img = Image.open(row.image_path).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
        imgs[i] = filter_fn(np.array(img))
        labels[i] = label_to_idx[row.dx]
    return imgs, labels

class ArrayDataset(Dataset):
    def __init__(self, imgs, labels):
        self.imgs = imgs
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        arr = self.imgs[idx].astype(np.float32) / 255.0
        tensor = torch.from_numpy(arr).permute(2, 0, 1)
        tensor = (tensor - IMAGENET_MEAN) / IMAGENET_STD
        return tensor, self.labels[idx]

def make_loader(imgs, labels, shuffle, batch_size=BATCH_SIZE):
    ds = ArrayDataset(imgs, labels)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                       num_workers=NUM_WORKERS, pin_memory=True,
                       persistent_workers=NUM_WORKERS > 0, drop_last=False)


In [ ]:
# ── Model builder ──────────────────────────────────────────────────────────
def get_model(name, num_classes=NUM_CLASSES):
    if name == "resnet101":
        m = torchvision.models.resnet101(weights=ResNet101_Weights.IMAGENET1K_V2)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    elif name == "densenet121":
        m = torchvision.models.densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1)
        m.classifier = nn.Linear(m.classifier.in_features, num_classes)
    else:
        raise ValueError(name)
    return m.to(DEVICE)


In [ ]:
# ── Training loop: AMP, grad clip, ReduceLROnPlateau(val macro-F1), early stop ──
def macro_f1_from_logits(logits, labels):
    preds = logits.argmax(1).cpu().numpy()
    return f1_score(labels.cpu().numpy(), preds, average="macro", zero_division=0)

def evaluate_loader(model, loader):
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEVICE.type == "cuda"):
        for x, y in loader:
            x = x.to(DEVICE, non_blocking=True)
            out = model(x)
            all_logits.append(out.float().cpu())
            all_labels.append(y)
    logits = torch.cat(all_logits)
    labels = torch.cat(all_labels)
    return macro_f1_from_logits(logits, labels), logits, labels

def train_model(name, tag, train_loader, val_loader):
    print(f"\n{'='*70}\nTraining {name} | filter={tag}\n{'='*70}")
    model = get_model(name)
    criterion = nn.CrossEntropyLoss(weight=class_weights_t)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)
    scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == "cuda")

    best_f1 = -1.0
    best_state = None
    epochs_no_improve = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        t0 = time.time()
        running_loss = 0.0
        for x, y in train_loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEVICE.type == "cuda"):
                out = model(x)
                loss = criterion(out, y)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item() * x.size(0)

        train_loss = running_loss / len(train_loader.dataset)
        val_f1, _, _ = evaluate_loader(model, val_loader)
        scheduler.step(val_f1)
        dt = time.time() - t0
        print(f"Epoch {epoch:02d} | train_loss {train_loss:.4f} | val_macro_f1 {val_f1:.4f} | {dt:.1f}s")

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= PATIENCE:
                print(f"Early stopping at epoch {epoch} (best val_macro_f1={best_f1:.4f})")
                break

    model.load_state_dict(best_state)
    ckpt_path = WORK_DIR / f"{name}_{tag}_best.pt"
    torch.save(best_state, ckpt_path)
    print(f"Best val macro-F1 for {name}/{tag}: {best_f1:.4f} (saved to {ckpt_path})")
    return model


In [ ]:
# ── Metrics helper ─────────────────────────────────────────────────────────
def compute_metrics(y_true, y_pred, y_proba, n_classes=NUM_CLASSES):
    acc  = accuracy_score(y_true, y_pred) * 100
    prec = precision_score(y_true, y_pred, average="weighted", zero_division=0) * 100
    rec  = recall_score(y_true, y_pred, average="weighted", zero_division=0) * 100
    f1w  = f1_score(y_true, y_pred, average="weighted", zero_division=0) * 100
    f1m  = f1_score(y_true, y_pred, average="macro", zero_division=0) * 100
    try:
        y_true_oh = label_binarize(y_true, classes=list(range(n_classes)))
        auc = roc_auc_score(y_true_oh, y_proba, average="macro", multi_class="ovr") * 100
    except Exception:
        auc = float("nan")
    return {"Accuracy": acc, "Precision": prec, "Recall": rec,
            "F1-score": f1w, "Macro-F1": f1m, "AUC": auc}

@torch.no_grad()
def extract_features(extractor, loader):
    feats, labels = [], []
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEVICE.type == "cuda"):
        for x, y in loader:
            x = x.to(DEVICE, non_blocking=True)
            f = extractor(x)
            feats.append(f.float().cpu().numpy())
            labels.append(y.numpy())
    return np.concatenate(feats), np.concatenate(labels)


In [ ]:
# ── Main loop: retrain all 3 models fresh, once per filter ────────────────
results = []
run_times = {}

for filter_name, filter_fn in FILTERS.items():
    t_filter_start = time.time()
    print(f"\n{'#'*70}\n# FILTER: {filter_name}\n{'#'*70}")

    # -- Materialize this filter's train/val/test arrays once --
    train_imgs, train_lbls = build_array_split(train_df, filter_fn)
    val_imgs, val_lbls     = build_array_split(val_df, filter_fn)
    test_imgs, test_lbls   = build_array_split(test_df, filter_fn)

    train_loader_f = make_loader(train_imgs, train_lbls, shuffle=True)
    val_loader_f   = make_loader(val_imgs, val_lbls, shuffle=False)
    test_loader_f  = make_loader(test_imgs, test_lbls, shuffle=False)

    tag = filter_name.lower().replace(" ", "_")

    # -- Best Model 2: ResNet101, fine-tuned on THIS filter's data --
    resnet101_model = train_model("resnet101", tag, train_loader_f, val_loader_f)
    _, logits, labels = evaluate_loader(resnet101_model, test_loader_f)
    probs = F.softmax(logits, dim=1).numpy()
    preds = probs.argmax(1)
    m = compute_metrics(labels.numpy(), preds, probs)
    results.append({"Model": "Best Model 2 (ResNet101, fine-tuned)", "Filter": filter_name, **m})

    # -- Best Model 3: DenseNet121, fine-tuned on THIS filter's data --
    densenet121_model = train_model("densenet121", tag, train_loader_f, val_loader_f)
    _, logits, labels = evaluate_loader(densenet121_model, test_loader_f)
    probs = F.softmax(logits, dim=1).numpy()
    preds = probs.argmax(1)
    m = compute_metrics(labels.numpy(), preds, probs)
    results.append({"Model": "Best Model 3 (DenseNet121, fine-tuned)", "Filter": filter_name, **m})

    # -- Best Model 1: THIS filter's ResNet101 deep features -> XGBoost --
    feat_extractor = copy.deepcopy(resnet101_model)
    feat_extractor.fc = nn.Identity()
    feat_extractor.eval().to(DEVICE)

    trainval_imgs = np.concatenate([train_imgs, val_imgs])
    trainval_lbls = np.concatenate([train_lbls, val_lbls])
    trainval_loader_f = make_loader(trainval_imgs, trainval_lbls, shuffle=False)

    X_trainval, y_trainval = extract_features(feat_extractor, trainval_loader_f)
    X_test, y_test = extract_features(feat_extractor, test_loader_f)

    scaler = StandardScaler().fit(X_trainval)
    X_trainval_s = scaler.transform(X_trainval)
    X_test_s = scaler.transform(X_test)

    xgb_clf = xgb.XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        objective="multi:softprob", num_class=NUM_CLASSES,
        eval_metric="mlogloss", tree_method="hist",
        device="cuda" if DEVICE.type == "cuda" else "cpu",
        random_state=SEED, n_jobs=-1,
    )
    xgb_clf.fit(X_trainval_s, y_trainval)

    probs = xgb_clf.predict_proba(X_test_s)
    preds = probs.argmax(1)
    m = compute_metrics(y_test, preds, probs)
    results.append({"Model": "Best Model 1 (ResNet101 features + XGBoost)", "Filter": filter_name, **m})

    run_times[filter_name] = time.time() - t_filter_start
    print(f"\nFilter '{filter_name}' done in {run_times[filter_name]/60:.1f} min")

    # Free memory before next filter
    del train_imgs, val_imgs, test_imgs, trainval_imgs
    del resnet101_model, densenet121_model, feat_extractor
    torch.cuda.empty_cache()



######################################################################
# FILTER: No Filter
######################################################################

Training resnet101 | filter=no_filter
Downloading: "https://download.pytorch.org/models/resnet101-cd907fc2.pth" to /root/.cache/torch/hub/checkpoints/resnet101-cd907fc2.pth


100%|██████████| 171M/171M [00:00<00:00, 180MB/s]


Epoch 01 | train_loss 1.6375 | val_macro_f1 0.4807 | 76.4s
Epoch 02 | train_loss 0.7548 | val_macro_f1 0.5922 | 54.1s
Epoch 03 | train_loss 0.3293 | val_macro_f1 0.6367 | 52.5s
Epoch 04 | train_loss 0.1321 | val_macro_f1 0.6587 | 52.6s
Epoch 05 | train_loss 0.0477 | val_macro_f1 0.6559 | 52.4s
Epoch 06 | train_loss 0.0285 | val_macro_f1 0.6877 | 52.5s
Epoch 07 | train_loss 0.0189 | val_macro_f1 0.7132 | 52.4s
Epoch 08 | train_loss 0.0126 | val_macro_f1 0.6190 | 52.3s
Epoch 09 | train_loss 0.0118 | val_macro_f1 0.6491 | 52.8s
Epoch 10 | train_loss 0.0124 | val_macro_f1 0.6687 | 52.6s
Epoch 11 | train_loss 0.0060 | val_macro_f1 0.6803 | 52.3s
Epoch 12 | train_loss 0.0033 | val_macro_f1 0.6760 | 52.2s
Epoch 13 | train_loss 0.0016 | val_macro_f1 0.6899 | 52.2s
Early stopping at epoch 13 (best val_macro_f1=0.7132)
Best val macro-F1 for resnet101/no_filter: 0.7132 (saved to /kaggle/working/resnet101_no_filter_best.pt)

Training densenet121 | filter=no_filter
Downloading: "https://download.py

100%|██████████| 30.8M/30.8M [00:00<00:00, 139MB/s]


Epoch 01 | train_loss 1.2959 | val_macro_f1 0.5809 | 61.7s
Epoch 02 | train_loss 0.6042 | val_macro_f1 0.6602 | 39.2s
Epoch 03 | train_loss 0.3226 | val_macro_f1 0.6650 | 39.3s
Epoch 04 | train_loss 0.1597 | val_macro_f1 0.6766 | 39.1s
Epoch 05 | train_loss 0.0740 | val_macro_f1 0.7124 | 39.1s
Epoch 06 | train_loss 0.0397 | val_macro_f1 0.6949 | 39.1s
Epoch 07 | train_loss 0.0265 | val_macro_f1 0.7264 | 39.1s
Epoch 08 | train_loss 0.0158 | val_macro_f1 0.6823 | 39.2s
Epoch 09 | train_loss 0.0113 | val_macro_f1 0.6903 | 39.3s
Epoch 10 | train_loss 0.0114 | val_macro_f1 0.6898 | 39.0s
Epoch 11 | train_loss 0.0072 | val_macro_f1 0.7134 | 39.3s
Epoch 12 | train_loss 0.0045 | val_macro_f1 0.6963 | 39.6s
Epoch 13 | train_loss 0.0020 | val_macro_f1 0.6920 | 40.5s
Early stopping at epoch 13 (best val_macro_f1=0.7264)
Best val macro-F1 for densenet121/no_filter: 0.7264 (saved to /kaggle/working/densenet121_no_filter_best.pt)

Filter 'No Filter' done in 24.1 min

################################

In [ ]:
# ── Assemble the results table ─────────────────────────────────────────────
results_df = pd.DataFrame(results)

filter_order = list(FILTERS.keys())
model_order = ["Best Model 1 (ResNet101 features + XGBoost)",
               "Best Model 2 (ResNet101, fine-tuned)",
               "Best Model 3 (DenseNet121, fine-tuned)"]
results_df["Model"] = pd.Categorical(results_df["Model"], categories=model_order, ordered=True)
results_df["Filter"] = pd.Categorical(results_df["Filter"], categories=filter_order, ordered=True)
results_df = results_df.sort_values(["Model", "Filter"]).reset_index(drop=True)
results_df = results_df.round(2)
results_df.to_csv(WORK_DIR / "filter_retrain_results.csv", index=False)

print("Per-filter wall time (minutes):")
for k, v in run_times.items():
    print(f"  {k:12s} {v/60:.1f}")

results_df


Per-filter wall time (minutes):
  No Filter    24.1
  Average      24.2
  Gaussian     21.1
  Median       26.4
  Sharpening   22.1
  Sobel        22.1


,Model,Filter,Accuracy,Precision,Recall,F1-score,Macro-F1,AUC
0,Best Model 1 (ResNet101 features + XGBoost),No Filter,81.21,80.63,81.21,80.80,62.78,93.65
1,Best Model 1 (ResNet101 features + XGBoost),Average,80.20,79.47,80.20,79.56,63.78,93.23
2,Best Model 1 (ResNet101 features + XGBoost),Gaussian,79.70,78.74,79.70,78.97,64.12,93.59
3,Best Model 1 (ResNet101 features + XGBoost),Median,79.39,78.51,79.39,78.80,60.66,93.17
4,Best Model 1 (ResNet101 features + XGBoost),Sharpening,82.12,81.63,82.12,81.75,65.01,94.13
5,Best Model 1 (ResNet101 features + XGBoost),Sobel,73.64,70.25,73.64,71.33,42.94,86.48
6,"Best Model 2 (ResNet101, fine-tuned)",No Filter,79.29,80.85,79.29,79.71,60.71,90.97
7,"Best Model 2 (ResNet101, fine-tuned)",Average,79.19,79.68,79.19,79.20,63.30,90.10
8,"Best Model 2 (ResNet101, fine-tuned)",Gaussian,73.94,81.03,73.94,76.28,61.25,91.46
9,"Best Model 2 (ResNet101, fine-tuned)",Median,79.60,78.63,79.60,78.87,61.51,90.61


## Additional Analysis

- Reproduction check for the **No Filter** row of each model vs. the
  original report (same split/hyperparams, retrained from scratch here).
- Which filter is the *best preprocessing choice* for each model (highest
  accuracy), vs. which one hurts it most.


In [ ]:
# ── Additional analysis ────────────────────────────────────────────────────
baseline_reported = {
    "Best Model 1 (ResNet101 features + XGBoost)": 82.93,
    "Best Model 2 (ResNet101, fine-tuned)": 80.91,
    "Best Model 3 (DenseNet121, fine-tuned)": 82.22,
}

print("Baseline reproduction check (No Filter):")
for model_name, reported_acc in baseline_reported.items():
    row = results_df[(results_df.Model == model_name) & (results_df.Filter == "No Filter")]
    got = row["Accuracy"].values[0]
    print(f"  {model_name:45s} reported={reported_acc:6.2f}%  reproduced={got:6.2f}%  Δ={got - reported_acc:+.2f}")

print("\nBest / worst filter per model (by accuracy):")
for model_name in model_order:
    sub = results_df[results_df.Model == model_name].set_index("Filter")["Accuracy"]
    best_filter, worst_filter = sub.idxmax(), sub.idxmin()
    print(f"\n{model_name}")
    print(f"  best:  {best_filter:12s} {sub[best_filter]:.2f}%")
    print(f"  worst: {worst_filter:12s} {sub[worst_filter]:.2f}%")


Baseline reproduction check (No Filter):
  Best Model 1 (ResNet101 features + XGBoost)   reported= 82.93%  reproduced= 81.21%  Δ=-1.72
  Best Model 2 (ResNet101, fine-tuned)          reported= 80.91%  reproduced= 79.29%  Δ=-1.62
  Best Model 3 (DenseNet121, fine-tuned)        reported= 82.22%  reproduced= 81.31%  Δ=-0.91

Best / worst filter per model (by accuracy):

Best Model 1 (ResNet101 features + XGBoost)
  best:  Sharpening   82.12%
  worst: Sobel        73.64%

Best Model 2 (ResNet101, fine-tuned)
  best:  Sharpening   81.72%
  worst: Sobel        65.76%

Best Model 3 (DenseNet121, fine-tuned)
  best:  No Filter    81.31%
  worst: Sobel        73.84%
